In [1]:
import os

os.environ["CC"] = "/usr/bin/gcc"
os.environ["CXX"] = "/usr/bin/g++"

import cmdstanpy
cmdstanpy.install_cmdstan(verbose=True)

/u/zwu1/.conda/envs/ou/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CmdStan install directory: /u/zwu1/.cmdstan
CmdStan version 2.38.0 already installed
Test model compilation

--- Translating Stan model to C++ code ---
bin/stanc  --o=examples/bernoulli/bernoulli.hpp examples/bernoulli/bernoulli.stan

--- Compiling C++ code ---
/usr/bin/g++ -std=c++17 -pthread -D_REENTRANT -Wno-sign-compare -Wno-ignored-attributes -Wno-class-memaccess      -I stan/lib/stan_math/lib/tbb_2020.3/include    -O3 -I src -I stan/src -I stan/lib/rapidjson_1.1.0/ -I lib/CLI11-1.9.1/ -I stan/lib/stan_math/ -I stan/lib/stan_math/lib/eigen_3.4.0 -I stan/lib/stan_math/lib/boost_1.87.0 -I stan/lib/stan_math/lib/sundials_6.1.1/include -I stan/lib/stan_math/lib/sundials_6.1.1/src/sundials    -DBOOST_DISABLE_ASSERTS          -c -Wno-ignored-attributes   -x c++ -o examples/bernoulli/bernoulli.o examples/bernoulli/bernoulli.hpp

--- Linking model ---
/usr/bin/g++ -std=c++17 -pthread -D_REENTRANT -Wno-sign-compare -Wno-ignored-attributes -Wno-class-memaccess      -I stan/lib/stan_math/lib

True

In [2]:
import numpy as np
import scipy.linalg as la
import pandas as pd
import time
import os
from cmdstanpy import CmdStanModel

# ============================================================================
# 1. Define and Compile the Stan Models
# ============================================================================

# --- MODEL A: Our Innovative Joint GMRF Sampler ---
stan_joint = """
functions {
  matrix solve_lyapunov(matrix Gamma, matrix Sigma, int R) {
    matrix[R*R, R*R] K; vector[R*R] vec_Sigma; vector[R*R] vec_Omega; matrix[R, R] Omega;
    for (i in 1:R) { for (j in 1:R) { for (k in 1:R) { for (l in 1:R) {
            int row_idx = (j - 1) * R + i; int col_idx = (l - 1) * R + k;
            K[row_idx, col_idx] = (i == k ? Gamma[j, l] : 0.0) + (j == l ? Gamma[i, k] : 0.0);
    } } } }
    for (i in 1:R) { for (j in 1:R) { vec_Sigma[(j - 1) * R + i] = Sigma[i, j]; } }
    vec_Omega = K \\ vec_Sigma;
    for (i in 1:R) { for (j in 1:R) { Omega[i, j] = vec_Omega[(j - 1) * R + i]; } }
    return Omega;
  }
}
data {
    int N; int Nsub; int K; int R;
    array[N] int ID; array[Nsub] int cumu; array[Nsub] int repme;
    array[N, K] int Y; vector[N] deltat; int ncate;
}
parameters {
    real theta1; real theta2; real theta3;
    ordered[ncate - 1] theta4; ordered[ncate - 1] theta5; 
    ordered[ncate - 1] theta6; ordered[ncate - 1] theta7;
    vector<lower=1e-6>[K - 2] lambda_free;
    matrix[Nsub, K] b_raw; vector<lower=1e-6>[K] sigma_bk;
    
    // S + A Structural parameters
    cholesky_factor_corr[R] L_S_corr; vector<lower=0>[R] L_S_scale;
    vector[R * (R - 1) / 2] a_low;
    cholesky_factor_corr[R] L_Sigma_corr; vector<lower=0>[R] L_Sigma_scale;
    
    // Non-Centered Latent State
    matrix[R, N] xi_raw; 
}
transformed parameters {
    matrix[Nsub, K] b; vector[K] lambda; matrix[R, N] xi;
    matrix[R, R] Gamma; matrix[R, R] Sigma; matrix[R, R] Omega;
    
    lambda[1] = 1.0; lambda[2] = lambda_free[1]; lambda[3] = lambda_free[2];
    lambda[4] = 1.0; lambda[5] = lambda_free[3]; lambda[6] = lambda_free[4]; lambda[7] = lambda_free[5];

    for (i in 1:Nsub) { for (k in 1:K) { b[i, k] = b_raw[i, k] * sigma_bk[k]; } }
    
    Gamma = multiply_lower_tri_self_transpose(diag_pre_multiply(L_S_scale, L_S_corr));
    Gamma[2, 1] = Gamma[2, 1] - a_low[1]; Gamma[1, 2] = Gamma[1, 2] + a_low[1]; // S + A
    
    Sigma = multiply_lower_tri_self_transpose(diag_pre_multiply(L_Sigma_scale, L_Sigma_corr));
    Omega = solve_lyapunov(Gamma, Sigma, R);
    Omega = 0.5 * (Omega + Omega'); Omega = add_diag(Omega, 1e-5);
    
    {
        matrix[R, R] L_Omega = cholesky_decompose(Omega);
        for (i in 1:Nsub) {
            int start_idx = cumu[i] - repme[i] + 1;
            xi[:, start_idx] = L_Omega * xi_raw[:, start_idx];
            for (j in 2:repme[i]) {
                int k = start_idx + j - 1;
                matrix[R, R] Phi = matrix_exp(-deltat[k] * Gamma);
                matrix[R, R] Q = Omega - Phi * Omega * Phi';
                matrix[R, R] L_Q = cholesky_decompose(add_diag(0.5 * (Q + Q'), 1e-6));
                xi[:, k] = Phi * xi[:, k-1] + L_Q * xi_raw[:, k];
            }
        }
    }
}
model {
    lambda_free ~ normal(1, 2); sigma_bk ~ cauchy(0, 2); to_vector(b_raw) ~ std_normal();
    L_S_corr ~ lkj_corr_cholesky(2.0); L_S_scale ~ lognormal(0, 0.5); a_low ~ normal(0, 0.5);
    L_Sigma_corr ~ lkj_corr_cholesky(2.0); L_Sigma_scale ~ lognormal(0, 0.5);
    to_vector(xi_raw) ~ std_normal();

    for (i in 1:N) {
        int sub = ID[i];
        Y[i, 1] ~ bernoulli_logit(theta1 + lambda[1] * xi[1, i] + b[sub, 1]);
        Y[i, 2] ~ bernoulli_logit(theta2 + lambda[2] * xi[1, i] + b[sub, 2]);
        Y[i, 3] ~ bernoulli_logit(theta3 + lambda[3] * xi[1, i] + b[sub, 3]);
        Y[i, 4] ~ ordered_logistic(lambda[4] * xi[2, i] + b[sub, 4], theta4);
        Y[i, 5] ~ ordered_logistic(lambda[5] * xi[2, i] + b[sub, 5], theta5);
        Y[i, 6] ~ ordered_logistic(lambda[6] * xi[2, i] + b[sub, 6], theta6);
        Y[i, 7] ~ ordered_logistic(lambda[7] * xi[2, i] + b[sub, 7], theta7);
    }
}
"""

# --- MODEL B: The Standard Sequential Sampler (Centered & Constrained) ---
stan_sequential = """
data {
    int N; int Nsub; int K; int R;
    array[N] int ID; array[Nsub] int cumu; array[Nsub] int repme;
    array[N, K] int Y; vector[N] deltat; int ncate;
}
parameters {
    real theta1; real theta2; real theta3;
    ordered[ncate - 1] theta4; ordered[ncate - 1] theta5; 
    ordered[ncate - 1] theta6; ordered[ncate - 1] theta7;
    vector<lower=1e-6>[K - 2] lambda_free;
    matrix[Nsub, K] b_raw; vector<lower=1e-6>[K] sigma_bk;
    
    // Direct Parameterization of Gamma with Hard Constraints
    matrix[R, R] Gamma;
    cholesky_factor_corr[R] L_Sigma_corr; vector<lower=0>[R] L_Sigma_scale;
    
    // Centered Latent State (The Funnel Trap)
    matrix[R, N] xi; 
}
transformed parameters {
    matrix[Nsub, K] b; vector[K] lambda; matrix[R, R] Sigma;
    
    // Routh-Hurwitz Stability Constraints for R=2
    real<lower=0.0001> constraint1 = Gamma[1,1] + Gamma[2,2];
    real<lower=0.0001> constraint2 = Gamma[1,1]*Gamma[2,2] - Gamma[1,2]*Gamma[2,1];

    lambda[1] = 1.0; lambda[2] = lambda_free[1]; lambda[3] = lambda_free[2];
    lambda[4] = 1.0; lambda[5] = lambda_free[3]; lambda[6] = lambda_free[4]; lambda[7] = lambda_free[5];
    for (i in 1:Nsub) { for (k in 1:K) { b[i, k] = b_raw[i, k] * sigma_bk[k]; } }
    Sigma = multiply_lower_tri_self_transpose(diag_pre_multiply(L_Sigma_scale, L_Sigma_corr));
}
model {
    lambda_free ~ normal(1, 2); sigma_bk ~ cauchy(0, 2); to_vector(b_raw) ~ std_normal();
    to_vector(Gamma) ~ normal(0, 2); 
    L_Sigma_corr ~ lkj_corr_cholesky(2.0); L_Sigma_scale ~ lognormal(0, 0.5);

    // Solve Lyapunov for Time 1 stationary distribution (Vectorized manual approach for baseline)
    matrix[R*R, R*R] K_lyap; vector[R*R] vec_Sigma; vector[R*R] vec_Omega; matrix[R, R] Omega;
    for (i in 1:R) { for (j in 1:R) { for (k in 1:R) { for (l in 1:R) {
            int row_idx = (j - 1) * R + i; int col_idx = (l - 1) * R + k;
            K_lyap[row_idx, col_idx] = (i == k ? Gamma[j, l] : 0.0) + (j == l ? Gamma[i, k] : 0.0);
    } } } }
    for (i in 1:R) { for (j in 1:R) { vec_Sigma[(j - 1) * R + i] = Sigma[i, j]; } }
    vec_Omega = K_lyap \\ vec_Sigma;
    for (i in 1:R) { for (j in 1:R) { Omega[i, j] = vec_Omega[(j - 1) * R + i]; } }
    Omega = 0.5 * (Omega + Omega'); Omega = add_diag(Omega, 1e-5);

    // SEQUENTIAL SAMPLING
    for (i in 1:Nsub) {
        int start_idx = cumu[i] - repme[i] + 1;
        xi[:, start_idx] ~ multi_normal(rep_vector(0, R), Omega); // Time 1
        
        for (j in 2:repme[i]) {
            int k = start_idx + j - 1;
            matrix[R, R] Phi = matrix_exp(-deltat[k] * Gamma);
            matrix[R, R] Q = Omega - Phi * Omega * Phi';
            matrix[R, R] Q_sym = add_diag(0.5 * (Q + Q'), 1e-6);
            
            // The Centered Bottleneck
            xi[:, k] ~ multi_normal(Phi * xi[:, k-1], Q_sym); 
        }
    }

    for (i in 1:N) {
        int sub = ID[i];
        Y[i, 1] ~ bernoulli_logit(theta1 + lambda[1] * xi[1, i] + b[sub, 1]);
        Y[i, 2] ~ bernoulli_logit(theta2 + lambda[2] * xi[1, i] + b[sub, 2]);
        Y[i, 3] ~ bernoulli_logit(theta3 + lambda[3] * xi[1, i] + b[sub, 3]);
        Y[i, 4] ~ ordered_logistic(lambda[4] * xi[2, i] + b[sub, 4], theta4);
        Y[i, 5] ~ ordered_logistic(lambda[5] * xi[2, i] + b[sub, 5], theta5);
        Y[i, 6] ~ ordered_logistic(lambda[6] * xi[2, i] + b[sub, 6], theta6);
        Y[i, 7] ~ ordered_logistic(lambda[7] * xi[2, i] + b[sub, 7], theta7);
    }
}
"""

with open("model_joint.stan", "w") as f: f.write(stan_joint)
with open("model_sequential.stan", "w") as f: f.write(stan_sequential)

print("Compiling models (this may take a minute)...")
mod_joint = CmdStanModel(stan_file="model_joint.stan")
mod_seq = CmdStanModel(stan_file="model_sequential.stan")

# ============================================================================
# 2. Dynamic Data Generator
# ============================================================================
def generate_data(Nsub, obs_per_sub, Gamma_true):
    np.random.seed(42)
    N = Nsub * obs_per_sub; K = 7; R = 2; ncate = 4
    ID = np.repeat(np.arange(1, Nsub + 1), obs_per_sub)
    repme = np.full(Nsub, obs_per_sub); cumu = np.cumsum(repme)
    
    deltat = np.random.uniform(0.5, 3.0, size=N)
    deltat[::obs_per_sub] = 0.0 
    
    Sigma_true = np.array([[1.0, 0.2], [0.2, 1.0]])
    Omega_true = la.solve_continuous_lyapunov(Gamma_true, Sigma_true)
    
    xi_true = np.zeros((R, N))
    for i in range(Nsub):
        start_idx = cumu[i] - repme[i]
        xi_true[:, start_idx] = np.random.multivariate_normal(np.zeros(R), Omega_true)
        for j in range(1, repme[i]):
            idx = start_idx + j; dt = deltat[idx]
            Phi = la.expm(-Gamma_true * dt)
            Q_cond = 0.5 * ((Omega_true - Phi @ Omega_true @ Phi.T) + (Omega_true - Phi @ Omega_true @ Phi.T).T)
            xi_true[:, idx] = np.random.multivariate_normal(Phi @ xi_true[:, idx - 1], Q_cond)

    theta_bin = np.array([-0.5, 0.2, 0.8]); theta_ord = np.array([-1.5, 0.0, 1.5]) 
    lambda_true = np.array([1.0, 0.9, 1.1, 1.0, 0.8, 1.0, 1.4])
    b_true = np.random.normal(0, 0.3, size=(Nsub, K))
    
    Y = np.zeros((N, K), dtype=int)
    for i in range(N):
        sub_idx = ID[i] - 1
        for k in range(3):
            prob = 1.0 / (1.0 + np.exp(-(theta_bin[k] + lambda_true[k] * xi_true[0, i] + b_true[sub_idx, k])))
            Y[i, k] = np.random.binomial(1, prob)
        for k in range(3, 7):
            Z = lambda_true[k] * xi_true[1, i] + b_true[sub_idx, k] + np.random.logistic(0, 1)
            Y[i, k] = 1 if Z <= theta_ord[0] else (2 if Z <= theta_ord[1] else (3 if Z <= theta_ord[2] else 4))

    return {'N': N, 'Nsub': Nsub, 'K': K, 'R': R, 'ID': ID.tolist(), 'cumu': cumu.tolist(),
            'repme': repme.tolist(), 'Y': Y.tolist(), 'deltat': deltat.tolist(), 'ncate': ncate}

# ============================================================================
# 3. Experiment Runner
# ============================================================================
results = []

def run_experiment(exp_name, model, model_name, data, adapt_delta):
    print(f"Running {exp_name} | {model_name} ...")
    start_time = time.time()
    
    # Keeping iterations moderate to ensure the script completes in a reasonable time
    # The geometric failures in the sequential model will still trigger immediately.
    fit = model.sample(data=data, chains=2, iter_warmup=300, iter_sampling=300,
                       max_treedepth=10, adapt_delta=adapt_delta, show_console=False, show_progress=False)
    
    run_time = time.time() - start_time
    divergences = fit.divergences.sum()
    treedepths = (fit.method_variables()['treedepth__'] >= 10).sum()
    
    # Calculate efficiency based on a core parameter (e.g., loading lambda[2])
    ess = fit.summary().loc['lambda[2]', 'ESS_bulk']
    ess_per_sec = ess / run_time if run_time > 0 else 0
    
    results.append({
        'Experiment': exp_name,
        'Model': model_name,
        'Divergences': divergences,
        'Max Treedepths': treedepths,
        'ESS (Lambda 2)': round(ess, 1),
        'Time (s)': round(run_time, 1),
        'ESS / Sec': round(ess_per_sec, 2)
    })

# --- Define Experimental Conditions ---
Gamma_Standard = np.array([[1.5, 0.0], [0.0, 1.0]])
Gamma_Highly_Corr = np.array([[0.2, 0.0], [0.0, 0.2]])
Gamma_Cross_Lag = np.array([[1.0, 0.8], [-0.8, 1.0]])

experiments = [
    # Exp 1: Trajectory Length (Funnel Depth)
    ("Exp 1: Short Traj (T=10)", generate_data(30, 10, Gamma_Standard), 0.85),
    ("Exp 1: Long Traj (T=50)", generate_data(30, 50, Gamma_Standard), 0.85),
    
    # Exp 2: Autocorrelation (Stiffness)
    ("Exp 2: High Autocorr (Gamma=0.2)", generate_data(30, 30, Gamma_Highly_Corr), 0.85),
    
    # Exp 3: Multivariate (Cross-Lagged)
    ("Exp 3: High Cross-Lag", generate_data(30, 30, Gamma_Cross_Lag), 0.85),
    
    # Exp 4: Shrinking Step Size
    ("Exp 4: High Adapt Delta (0.99)", generate_data(30, 30, Gamma_Standard), 0.99)
]

# Run the suite
print("\nStarting Experimental Suite...\n")
for exp_name, data, adapt_delta in experiments:
    run_experiment(exp_name, mod_seq, "Baseline (Sequential)", data, adapt_delta)
    run_experiment(exp_name, mod_joint, "Innovation (Joint GMRF)", data, adapt_delta)

# ============================================================================
# 4. Print Summary
# ============================================================================
df_results = pd.DataFrame(results)
print("\n\n" + "="*80)
print("EXPERIMENTAL RESULTS: STANDARD SEQUENTIAL VS. JOINT GMRF")
print("="*80)
print(df_results.to_string(index=False))
print("="*80)

14:20:21 - cmdstanpy - INFO - compiling stan file /nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/model_joint.stan to exe file /nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/model_joint


Compiling models (this may take a minute)...


14:21:24 - cmdstanpy - INFO - compiled model executable: /nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/model_joint
14:21:24 - cmdstanpy - WARNING - Stan compiler has produced 1 warnings:
14:21:24 - cmdstanpy - WARNING - 
--- Translating Stan model to C++ code ---
bin/stanc --filename-in-msg=model_joint.stan --o=/nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/model_joint.hpp /nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/model_joint.stan
Warning in 'model_joint.stan', line 29, column 11 to column 22:
    Found int division:
        R * (R - 1) / 2
    Values will be rounded towards zero. If rounding is not desired you can
    write the division as
        R * (R - 1) / 2.0
    If rounding is intended please use the integer division operator %/%.

--- Compiling C++ code ---
/usr/bin/g++ -std=c++17 -pthread -D_REENTRANT -Wno-sign-compare -Wno-ignored-attributes -Wno-class-memaccess      -I stan/lib/stan_math/lib/tbb_2020.3/include    -O3 -I src -I stan/src -I stan/lib/rapidjson_1.1.0/ -I lib/CL


Starting Experimental Suite...

Running Exp 1: Short Traj (T=10) | Baseline (Sequential) ...


14:24:42 - cmdstanpy - INFO - Chain [2] done processing
14:24:57 - cmdstanpy - INFO - Chain [1] done processing
14:24:57 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: model_sequential_model_namespace::log_prob: constraint1 is -1.50959, but must be greater than or equal to 0.000100 (in 'model_sequential.stan', line 25, column 4 to column 61)
	Exception: ordered_logistic: Cut-points is not a valid ordered vector. The element at 2 is -48.61, but should be greater than the previous element, -48.61 (in 'model_sequential.stan', line 73, column 8 to column 77)
	Exception: ordered_logistic: Cut-points is not a valid ordered vector. The element at 2 is -49.4099, but should be greater than the previous element, -49.4099 (in 'model_sequential.stan', line 73, column 8 to column 77)
	Exception: lkj_corr_cholesky_lpdf: Random variable[2] is 0, but must be positive! (in 'model_sequential.stan', line 36, column 4 to column 42)
	Exception: ordered_logistic: Cut-points is not a val

...

Running Exp 4: High Adapt Delta (0.99) | Baseline (Sequential) ...


15:31:15 - cmdstanpy - INFO - Chain [2] done processing
15:34:41 - cmdstanpy - INFO - Chain [1] done processing
15:34:41 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: model_sequential_model_namespace::log_prob: constraint1 is -0.404273, but must be greater than or equal to 0.000100 (in 'model_sequential.stan', line 25, column 4 to column 61)
	Exception: model_sequential_model_namespace::log_prob: constraint2 is -2.91842, but must be greater than or equal to 0.000100 (in 'model_sequential.stan', line 26, column 4 to column 83)
	Exception: model_sequential_model_namespace::log_prob: constraint2 is -1.436, but must be greater than or equal to 0.000100 (in 'model_sequential.stan', line 26, column 4 to column 83)
	Exception: model_sequential_model_namespace::log_prob: constraint1 is -0.718285, but must be greater than or equal to 0.000100 (in 'model_sequential.stan', line 25, column 4 to column 61)
	Exception: model_sequential_model_namespace::log_prob: constraint1 is 

Running Exp 4: High Adapt Delta (0.99) | Innovation (Joint GMRF) ...


15:45:23 - cmdstanpy - INFO - Chain [2] done processing
15:47:46 - cmdstanpy - INFO - Chain [1] done processing
15:47:46 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: cholesky_decompose: A is not symmetric. A[1,2] = -nan, but A[2,1] = -nan (in 'model_joint.stan', line 52, column 8 to column 57)
	Exception: cholesky_decompose: A is not symmetric. A[1,2] = -nan, but A[2,1] = -nan (in 'model_joint.stan', line 52, column 8 to column 57)
	Exception: cholesky_decompose: A is not symmetric. A[1,2] = inf, but A[2,1] = inf (in 'model_joint.stan', line 52, column 8 to column 57)
	Exception: cholesky_decompose: A is not symmetric. A[1,2] = inf, but A[2,1] = inf (in 'model_joint.stan', line 52, column 8 to column 57)
	Exception: cholesky_decompose: A is not symmetric. A[1,2] = -nan, but A[2,1] = -nan (in 'model_joint.stan', line 52, column 8 to column 57)
	Exception: ordered_logistic: Cut-points is not a valid ordered vector. The element at 2 is -186.001, but should be greate



EXPERIMENTAL RESULTS: STANDARD SEQUENTIAL VS. JOINT GMRF
                      Experiment                   Model  Divergences  Max Treedepths  ESS (Lambda 2)  Time (s)  ESS / Sec
        Exp 1: Short Traj (T=10)   Baseline (Sequential)            1               0           113.1     151.5       0.75
        Exp 1: Short Traj (T=10) Innovation (Joint GMRF)            0               0           268.6      88.3       3.04
         Exp 1: Long Traj (T=50)   Baseline (Sequential)            0               0             9.0     805.2       0.01
         Exp 1: Long Traj (T=50) Innovation (Joint GMRF)            0               0           116.3     562.8       0.21
Exp 2: High Autocorr (Gamma=0.2)   Baseline (Sequential)            0               0            17.9     301.9       0.06
Exp 2: High Autocorr (Gamma=0.2) Innovation (Joint GMRF)            0               0           672.3     458.8       1.47
           Exp 3: High Cross-Lag   Baseline (Sequential)            0           